# For 12.842 Only: PSet #2 Build a cloud!

> This notebook is adapted from Chapter 26 in the [Climate Laboratory Book](https://brian-rose.github.io/ClimateLaboratoryBook/home.html) by Brian E. J. Rose, University at Albany




## Installing climlab in Google Colab (again... sorry!)

We will install climlab again since we've opened a new colab notebook

In [ ]:
# Install climlab with conda colab (this might take awhile!)
%%capture
!pip install -q condacolab
import condacolab
condacolab.install()
!conda install -c conda-forge climlab

# IMPORTANT! The Python kernel will automatically be restarted for changes to be applied.
# So if you see a message saying "Your session crashed for an unknown reason",
# you can safely ignore it and scroll down to run the next cell!

## Import modules

In [ ]:
# Import climlab and check installation
import climlab
climlab.__version__

In [ ]:
# Import other modules
from climlab import constants as const
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import statistics as stat

## First create the clear-sky model

This is the same model as 'rcm_H2Ofeedback' in PSet #2 Question 2 but we have a shorter timestep.

In [ ]:
# Temperatures in a single column
full_state = climlab.column_state(num_lev=20, water_depth=2.5)
temperature_state = {'Tatm':full_state.Tatm,'Ts':full_state.Ts}

#  Initialize a nearly dry column (small background stratospheric humidity)
q = np.ones_like(full_state.Tatm) * 5.E-6

#  Add specific_humidity to the state dictionary
full_state['q'] = q

#  ASYNCHRONOUS COUPLING -- the radiation uses a longer timestep than other processes

#  The top-level model
clear_rcm_H2O = climlab.TimeDependentProcess(state=full_state,
                              name='Clear-sky RCM',
                              timestep=0.5*const.seconds_per_hour)

#  Radiation coupled to water vapor
rad = climlab.radiation.RRTMG(state=temperature_state,
                              specific_humidity=full_state.q,
                              albedo=0.2,timestep=const.seconds_per_day,
                              )

#  Convection scheme -- water vapor is a state variable
conv = climlab.convection.EmanuelConvection(state=full_state,
                              timestep=0.5*const.seconds_per_hour)

#  Surface heat flux processes
shf = climlab.surface.SensibleHeatFlux(state=temperature_state, Cd=0.5E-3,
                              timestep=0.5*const.seconds_per_hour)

lhf = climlab.surface.LatentHeatFlux(state=full_state, Cd=0.5E-3,
                              timestep=0.5*const.seconds_per_hour)

#  Couple all the submodels together
clear_rcm_H2O.add_subprocess('Radiation', rad)
clear_rcm_H2O.add_subprocess('Convection', conv)
clear_rcm_H2O.add_subprocess('SHF', shf)
clear_rcm_H2O.add_subprocess('LHF', lhf)

print(clear_rcm_H2O)

## Now let's design some clouds!

Modify the code below as you wish.


In [ ]:
#  Initialize arrays of cloud properties
cldfrac = np.zeros_like(clear_rcm_H2O.state.Tatm)
r_liq = np.zeros_like(clear_rcm_H2O.state.Tatm) # Cloud water drop effective radius (microns)
r_ice = np.zeros_like(clear_rcm_H2O.state.Tatm) # Cloud ice crystal effective radius (microns)
clwp = np.zeros_like(clear_rcm_H2O.state.Tatm) # in-cloud liquid water path (g/m2)
ciwp = np.zeros_like(clear_rcm_H2O.state.Tatm) # in-cloud ice water path (g/m2)

#  Indices - modify this as you wish
high = 4  # corresponds to 225 hPa
low = 16   # corresponds to 825 hPa

#  A high, thin ice layer (cirrus cloud) - modify this as you wish
r_ice[high] = 14. 
ciwp[high] = 10.  
cldfrac[high] = 0.322

#  A low, thick, water cloud layer (stratus) - modify this as you wish
r_liq[low] = 14  
clwp[low] = 200.  
cldfrac[low] = 0.8

# wrap everything up in a dictionary
mycloud = {'cldfrac': cldfrac,
          'ciwp': ciwp,
          'clwp': clwp,
          'r_ice': r_ice,
          'r_liq': r_liq}

## Next, create the cloudy model

In [ ]:
# Temperatures in a single column
full_state = climlab.column_state(num_lev=20, water_depth=2.5)
temperature_state = {'Tatm':full_state.Tatm,'Ts':full_state.Ts}

#  Initialize a nearly dry column (small background stratospheric humidity)
q = np.ones_like(full_state.Tatm) * 5.E-6

#  Add specific_humidity to the state dictionary
full_state['q'] = q

#  ASYNCHRONOUS COUPLING -- the radiation uses a longer timestep than other processes

#  The top-level model
cloudy_rcm_H2O = climlab.TimeDependentProcess(state=full_state,
                              name='Cloudy RCM',
                              timestep=0.5*const.seconds_per_hour)

#  Radiation coupled to water vapor
rad = climlab.radiation.RRTMG(state=temperature_state,
                              specific_humidity=full_state.q,
                              albedo=0.2,timestep=const.seconds_per_day,
                              **mycloud # this is the key step!
                              )

#  Convection scheme -- water vapor is a state variable
conv = climlab.convection.EmanuelConvection(state=full_state,
                              timestep=0.5*const.seconds_per_hour)

#  Surface heat flux processes
shf = climlab.surface.SensibleHeatFlux(state=temperature_state, Cd=0.5E-3,
                              timestep=0.5*const.seconds_per_hour)

lhf = climlab.surface.LatentHeatFlux(state=full_state, Cd=0.5E-3,
                              timestep=0.5*const.seconds_per_hour)

#  Couple all the submodels together
cloudy_rcm_H2O.add_subprocess('Radiation', rad)
cloudy_rcm_H2O.add_subprocess('Convection', conv)
cloudy_rcm_H2O.add_subprocess('SHF', shf)
cloudy_rcm_H2O.add_subprocess('LHF', lhf)

print(cloudy_rcm_H2O)

**Question 2.7 Describe the properties you have selected for your cloud(s) and justify why you have assigned the vertical profile you have.**

**Question 2.8 How do the surface fluxes change between your clear-sky and cloudy model? Explain.**

**Question 2.9 How do the radiative fluxes change at the top of the atmosphere?**

You can check out other diagnostics by consulting the climlab documentation. Diagnostic fluxes are also available at pressure level interfaces.

In [ ]:
# Create plot vertical profile of cloud fraction here
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(cloudy_rcm_H2O.subprocess['Radiation'].cldfrac*100, cloudy_rcm_H2O.lev)
ax.invert_yaxis()
ax.set_yscale('log')
ax.set_ylim(1050, 100)
ax.set_yticks([i for i in range(1000, 99, -100)])
ax.set_yticklabels([i for i in range(1000, 99, -100)])
ax.set_ylabel('Pressure (hPa)')
ax.set_xlabel('Cloud Fraction (%)')
ax.grid()

In [ ]:
# Integrate both models forward 1 year & print fluxes here